In [1]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,stats_test,STT_INIT_POINTS, STT_N_ITER, VOT_INIT_POINTS, VOT_N_ITER
from scipy import stats

VOT config

In [2]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : (-10,10),
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-1.0,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.01),
           'weightWalk': (1,3),
           'weightWait': (1,3),
           'weightFeeder': (1,3), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [3]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : defaultAlpha_bounds,
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [4]:
stt_gen_synth_pop_baseline = [1.538359177843295,
 1.5295917791690679,
 1.5348367663998912,
 1.5345692372273643,
 1.5331592512372867,
 1.5217034952544148,
 1.5266326185511052,
 1.5334169124971047,
 1.5360137686306476,
 1.5339017221565026,
 1.5362097781749502,
 1.5268521109958817,
 1.5339477732745261,
 1.5272698871615034,
 1.5366142065675472,
 1.5363138577063769,
 1.5296998668830282,
 1.5197286106223606,
 1.5389714190118928,
 1.5313439434581202,
 1.537382448574118,
 1.5343533532165867,
 1.533794502058735,
 1.5379044402275153,
 1.540476837032657,
 1.5442550528088403,
 1.5314370912595778,
 1.5282256003186678,
 1.529788503142131,
 1.5278669293628444]

stt_base_pop_baseline = [8.078856646164647,
 8.069551887294185,
 8.075341126236319,
 8.077668179308883,
 8.072089261165532,
 8.08434725328873,
 8.070135626168561,
 8.07003999884651,
 8.074229639153051,
 8.073871153762488,
 8.076644709455191,
 8.067903638080667,
 8.074645258436167,
 8.07695697572408,
 8.069666279765226,
 8.073238056549226,
 8.075901298340282,
 8.074258667977556,
 8.066514722376704,
 8.072437841187186,
 8.07148870370448,
 8.069260779586159,
 8.067953580518305,
 8.074462078074248,
 8.072890298665042,
 8.076124680057342,
 8.07104170586535,
 8.066528599568002,
 8.070791356185456,
 8.081042173765757]

rq_3_4_results = [1.3753081319569687,
 1.3742488303399583,
 1.368884010723444,
 1.3901922800580793,
 1.361084756061178,
 1.3775440627186848,
 1.3790867291461066,
 1.3727054733959452,
 1.365534949443592,
 1.3863956306982672,
 1.369942893561297,
 1.3627206632717621,
 1.3818975650700178,
 1.3661656227329662,
 1.3582229687456173,
 1.3829006610569607,
 1.371984528786159,
 1.379002606109665,
 1.3698603823393452,
 1.3873749242135547,
 1.376056809933175,
 1.388874581056718,
 1.3668109468143579,
 1.3580711757372352,
 1.3754509132818438,
 1.3793868968859846,
 1.3736371235050056,
 1.3867188912473,
 1.3659524784652293,
 1.3679807519092755]

In [5]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [6]:

rq_4_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq4.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq4.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq4.csv',
    "other_args" : "--use_random_seed true"
}
rq4_1_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_4_1_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq4_1_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.35300 | 0.0       | 2.2032449 | -9.997712 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.723551 | -0.2      | -0.097624 | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -12.63315 | 0.0       | -0.826951 | 1.1737965 | -3.596130 | -3.018985 | 3.0074456 | -0.032706 | -0.749260 | -0.2      | -0.033486 | 2.7892133 | 1.1700884 | 1.0781095 | 1.0       | 1.0       |
| 3         | -8.335236 | 0.0       | -0.788923 | 9.1577906 | 0.3316528 | 1.9187711 | -1.844843 | -0.314185 | -0.332299 | -0.2      | -0.057472 | 2.9777221 | 2.4963313 | 1.5608

In [10]:
rq_1_config_no_seed = rq_4_1_config
rq_1_config_no_seed["other_args"] = "--use_random_seed false"
rq_4_1_results = []
for i in range(0,30):
    rq_4_1_results.append(- call_vot_simulation(config=rq_1_config_no_seed,**rq4_1_optimiser.max['params']))


In [11]:
rq_4_1_results

[4.107192103461534,
 4.089931333707253,
 4.104172307549402,
 4.099819286067584,
 4.092147380546496,
 4.113514427796199,
 4.089557072905693,
 4.104245321688032,
 4.0993655240625015,
 4.103798878625523,
 4.101218299896193,
 4.07609955762409,
 4.112416916315092,
 4.105474449116669,
 4.087083678727698,
 4.100771285685858,
 4.09217321538128,
 4.092633707903322,
 4.095278441379898,
 4.106089861485245,
 4.101872030124147,
 4.1000343691216345,
 4.082920879017399,
 4.079481500744014,
 4.098486818062991,
 4.092394595584255,
 4.087475965220166,
 4.086071301758151,
 4.0968919765924285,
 4.078628686540781]

In [9]:
stats_test(rq_3_4_results, rq_4_1_results)

P-value: 6.318523670293488e-58
Statistically significant difference.
t-statistic: -414.1541
p-value: 0.0000
95% CI of the difference: [-2.7219, -2.7017]
